# Kadane's Algorithm — Google L4 DSA Notes

## Solved Problems

1. [LC 53 — Maximum Subarray](https://leetcode.com/problems/maximum-subarray/)
2. [LC 918 — Maximum Sum Circular Subarray](https://leetcode.com/problems/maximum-sum-circular-subarray/)
3. [LC 152 — Maximum Product Subarray](https://leetcode.com/problems/maximum-product-subarray/)
4. [LC 1186 — Maximum Subarray Sum with One Deletion](https://leetcode.com/problems/maximum-subarray-sum-with-one-deletion/)
5. [LC 1749 — Maximum Absolute Sum of Any Subarray](https://leetcode.com/problems/maximum-absolute-sum-of-any-subarray/)
6. [LC 2606 — Find the Substring With the Maximum Cost](https://leetcode.com/problems/find-the-substring-with-the-maximum-cost/)

---

# Kadane Recognition

Ask:

> **Is the problem asking for the best value of a contiguous subarray/substring, and can the answer ending at the current position be summarized by a small state?**

Classic Kadane:

```text
best_ending = best answer for a subarray that MUST end here
answer      = best answer seen anywhere
```

Core decision:

```text
extend the previous subarray
OR
start a new subarray here
```


# 1. LC 53 — Maximum Subarray

**Pattern:** Classic Kadane

### Recognition

The problem asks for the **maximum sum of a contiguous subarray**.

The key question is:

> What is the best sum of a subarray that ends exactly at index `i`?

There are only two possibilities:

1. Start a new subarray at `i`.
2. Extend the best subarray ending at `i-1`.

Therefore:

`best_ending = max(nums[i], best_ending + nums[i])`

Then keep the global maximum.

### Mental Model

Do not ask:

> What is the best subarray among all subarrays?

Ask:

> What is the best subarray **ending here**?

That local state is enough to build the global answer.


In [ ]:
from typing import List

class Solution:
    def maxSubArray(self, nums: List[int]) -> int:
        best_ending = nums[0]
        ans = nums[0]

        for i in range(1, len(nums)):
            best_ending = max(nums[i], best_ending + nums[i])
            ans = max(ans, best_ending)

        return ans

### Complexity

- Time: **O(n)**
- Space: **O(1)**

### Key Lesson

**Maximum contiguous sum → Kadane.**

The most important state is:

`best_ending`



# 2. LC 918 — Maximum Sum Circular Subarray

**Pattern:** Kadane + Circular Array

### Recognition

The subarray can wrap from the end of the array back to the beginning.

There are two possibilities:

### Case 1 — Normal subarray

Use ordinary Kadane:

`best_max`

### Case 2 — Wrapping subarray

A wrapping subarray can be viewed as:

`total_sum - minimum_middle_subarray`

So:

`wrap_sum = total_sum - best_min`

Therefore:

`answer = max(best_max, total_sum - best_min)`

### Important Edge Case

If all numbers are negative:

`total_sum - best_min = 0`

But that represents choosing an empty subarray, which is invalid.

So if `best_max < 0`, return `best_max`.


In [ ]:
from typing import List

class Solution:
    def maxSubarraySumCircular(self, nums: List[int]) -> int:
        total = sum(nums)

        cur_max = best_max = nums[0]
        cur_min = best_min = nums[0]

        for num in nums[1:]:
            cur_max = max(num, cur_max + num)
            best_max = max(best_max, cur_max)

            cur_min = min(num, cur_min + num)
            best_min = min(best_min, cur_min)

        # If all numbers are negative, total - best_min would be 0,
        # which represents an empty subarray and is invalid.
        if best_max < 0:
            return best_max

        return max(best_max, total - best_min)

### Complexity

- Time: **O(n)**
- Space: **O(1)**

### Key Lesson

Circular maximum subarray:

> **normal maximum OR everything except the minimum middle segment**



# 3. LC 152 — Maximum Product Subarray

**Pattern:** Kadane + Two-State DP

### Recognition

This looks like Kadane, but addition has been replaced by multiplication.

The problem:

> Why isn't one state enough?

Because of negative numbers.

Example:

`[-2, 3, -4]`

A very negative product can become the maximum after multiplying by another negative.

Therefore we track:

- `cur_max` = maximum product ending here
- `cur_min` = minimum product ending here

For a new value `x`:

`new_max = max(x, x*old_max, x*old_min)`

`new_min = min(x, x*old_max, x*old_min)`

### Recognition Rule

> **Product + negative values → track both maximum and minimum.**


In [ ]:
from typing import List

class Solution:
    def maxProduct(self, nums: List[int]) -> int:
        cur_max = cur_min = nums[0]
        ans = nums[0]

        for num in nums[1:]:
            old_max = cur_max
            old_min = cur_min

            cur_max = max(num, num * old_max, num * old_min)
            cur_min = min(num, num * old_max, num * old_min)

            ans = max(ans, cur_max)

        return ans

### Complexity

- Time: **O(n)**
- Space: **O(1)**

### Key Lesson

The minimum state is not useless.

A state that looks bad now can become the best state later.


# 4. LC 1186 — Maximum Subarray Sum with One Deletion

**Pattern:** Kadane + DP State

### Recognition

This is almost ordinary Kadane, but we are allowed **one deletion**.

That means we need to know two things:

1. Best sum ending here with **no deletion**.
2. Best sum ending here with **one deletion already used**.

States:

```text
no_del
one_del
```

Transitions:

### No deletion

Either start here or extend:

`no_del = max(x, no_del + x)`

### One deletion

Two possibilities:

1. Delete the current element → previous `no_del`
2. Keep current element and deletion was already used → previous `one_del + x`

Therefore:

`one_del = max(previous_no_del, previous_one_del + x)`

### Recognition Rule

> **Kadane + one controlled exception → add one DP state.**


In [ ]:
from typing import List

class Solution:
    def maximumSum(self, arr: List[int]) -> int:
        no_del = arr[0]
        one_del = float("-inf")
        ans = arr[0]

        for x in arr[1:]:
            prev_no_del = no_del
            prev_one_del = one_del

            no_del = max(x, prev_no_del + x)
            one_del = max(prev_no_del, prev_one_del + x)

            ans = max(ans, no_del, one_del)

        return ans

### Complexity

- Time: **O(n)**
- Space: **O(1)**

### Key Lesson

Many Kadane variations can be solved by asking:

> **What extra information do I need because of the new rule?**

Then add the smallest possible state.


# 5. LC 1749 — Maximum Absolute Sum of Any Subarray

**Pattern:** Kadane

### Recognition

We need:

`max(abs(subarray_sum))`

A large absolute value can come from either:

- a very large positive subarray sum
- a very large negative subarray sum

So calculate both:

`best_max = maximum subarray sum`

`best_min = minimum subarray sum`

Then:

`answer = max(best_max, -best_min)`

This is effectively running Kadane twice: once for maximum and once for minimum.


In [ ]:
from typing import List

class Solution:
    def maxAbsoluteSum(self, nums: List[int]) -> int:
        cur_max = best_max = 0
        cur_min = best_min = 0

        for num in nums:
            cur_max = max(0, cur_max + num)
            best_max = max(best_max, cur_max)

            cur_min = min(0, cur_min + num)
            best_min = min(best_min, cur_min)

        return max(best_max, -best_min)

### Complexity

- Time: **O(n)**
- Space: **O(1)**

### Key Lesson

**Absolute value → track both extremes.**

`max(abs(sum)) = max(max_sum, -min_sum)`


# 6. LC 2606 — Find the Substring With the Maximum Cost

**Pattern:** Transform + Kadane

### Recognition

The problem assigns a numerical value to each character and asks for the maximum-cost substring.

First transform:

```text
character → numeric cost
```

Now the problem becomes:

> Find the maximum-sum contiguous subarray.

That's exactly Kadane.

### Important Pattern Recognition Lesson

Sometimes the input doesn't look like an array of numbers.

Ask:

> **Can I transform every element into a value so that the problem becomes a known pattern?**

Here:

`string → character values → maximum subarray`


In [ ]:
class Solution:
    def maximumCostSubstring(
        self,
        s: str,
        chars: str,
        vals: list[int]
    ) -> int:
        value = {chr(ord('a') + i): i + 1 for i in range(26)}

        for ch, val in zip(chars, vals):
            value[ch] = val

        cur = 0
        ans = 0

        for ch in s:
            cur = max(0, cur + value[ch])
            ans = max(ans, cur)

        return ans

### Complexity

- Time: **O(n + 26)** → effectively O(n)
- Space: **O(26)** → effectively O(1)

### Key Lesson

**Transform first, then recognize the underlying pattern.**



# Kadane Master Cheat Sheet

## 1. Classic Maximum

```python
cur = max(x, cur + x)
ans = max(ans, cur)
```

**Recognition:** maximum contiguous sum.

---

## 2. Minimum

Mirror Kadane:

```python
cur = min(x, cur + x)
```

**Recognition:** minimum contiguous sum.

---

## 3. Circular

```text
max(normal_max, total_sum - minimum_subarray)
```

**Recognition:** circular maximum subarray.

---

## 4. Product

Track two states:

```text
cur_max
cur_min
```

because a negative can flip them.

**Recognition:** maximum product + negative numbers.

---

## 5. One Deletion / One Exception

Add another state:

```text
no_deletion
one_deletion
```

**Recognition:** Kadane + one controlled modification.

---

## 6. Absolute Sum

Track both:

```text
maximum subarray
minimum subarray
```

Then:

```text
max(max_sum, -min_sum)
```

**Recognition:** maximum absolute subarray sum.

---

## 7. Transform + Kadane

Convert each element into a numerical value, then run Kadane.

**Recognition:** custom scoring of contiguous substring/subarray.

---

# Final Recognition Question

When you see a new problem, ask:

> **Is there a contiguous segment whose best answer can be summarized by a small state ending at the current position?**

If yes, try Kadane.

Then ask:

1. Is it **sum** or **product**?
2. Is it **maximum**, **minimum**, or **absolute**?
3. Is the array **circular**?
4. Is there a **deletion/modification**?
5. Can I **transform** the input into numeric values?

These questions usually reveal which Kadane variant you need.
